In [1]:
# ignore the unnecessary warning messages
import warnings
warnings.filterwarnings("ignore")

In [2]:
# if document is too long, it need to go-through chunking process

# example chunked document
# using multiple user question

import json

with open("document_chunks_and_questions.json", "r", encoding="utf-8") as file:
    data = json.load(file)

examples_chunks_list = data['examples_chunks_list']
user_questions = data['user_questions']

In [3]:
# Function to calculate cosine similarity scores between user-question embeddings and document embeddings

from sklearn.metrics.pairwise import cosine_similarity

def similarity_scorer(database_embedding, question_embeddings): 

    for question_embedding, user_question in zip(question_embeddings, user_questions):
        scores = cosine_similarity(
            [question_embedding],
            database_embedding
        )[0]

        print(f"User-Question: {user_question}")
        
        for doc, score in zip(examples_chunks_list, scores):
            print(f"Similarity Score: {float(score) * 100:.2f}% - {doc.split('.')[0]}")

        print("\n")

In [4]:
# Using Sentence Transformers, we can download and run compatible open-weight models locally.

# Before downloading larger models, check whether the system has sufficient RAM, VRAM, and processing capability.

# Hugging Face Hub supports both authenticated and unauthenticated requests for accessing public models.

# Unauthenticated requests have lower rate limits. Setting an HF_TOKEN enables higher rate limits and can improve download reliability and speed.

# default cache location:
# Windows: C:\Users\<username>\.cache\huggingface\hub
# Linux: ~/.cache/huggingface/hub
# Mac: ~/.cache/huggingface/hub

from sentence_transformers import SentenceTransformer

sentence_transformer_model = SentenceTransformer("all-MiniLM-L6-v2")

sentence_transformer_embeddings = sentence_transformer_model.encode(examples_chunks_list)

print("shape : ", sentence_transformer_embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2958.02it/s]


shape :  (4, 384)


In [5]:
# embedding the user-question

transformer_questions_embedding = sentence_transformer_model.encode(user_questions)

print("shape : ", transformer_questions_embedding.shape)

shape :  (3, 384)


In [6]:
# finding the cosine similarity
similarity_scorer(sentence_transformer_embeddings, transformer_questions_embedding)

User-Question: What is Python and what is it used for?
Similarity Score: 83.06% - Python is a high-level programming language
Similarity Score: 17.58% -  RAG stands for Retrieval-Augmented Generation
Similarity Score: 21.20% -  Vector databases store numerical representations called embeddings
Similarity Score: 24.54% - Machine learning is a field of artificial intelligence


User-Question: How does Retrieval-Augmented Generation (RAG) work?
Similarity Score: -4.33% - Python is a high-level programming language
Similarity Score: 66.65% -  RAG stands for Retrieval-Augmented Generation
Similarity Score: 22.18% -  Vector databases store numerical representations called embeddings
Similarity Score: 8.90% - Machine learning is a field of artificial intelligence


User-Question: What are embeddings and how do vector databases use them?
Similarity Score: 9.22% - Python is a high-level programming language
Similarity Score: 23.21% -  RAG stands for Retrieval-Augmented Generation
Similarity Sco

In [7]:
# same process using ollama locally running models and sdk
# creating the client - make the connections with ollama endpoints
import ollama

ollama_client = ollama.Client(
    host="http://127.0.0.1:11434"
)

print("Ollama Client connection is successful")

Ollama Client connection is successful


In [8]:
ollama_embedding_model = ollama_client.embed(
    model="nomic-embed-text:latest",
    input=examples_chunks_list
)

ollama_embedding = ollama_embedding_model['embeddings']

print(f"shape: {len(ollama_embedding)}, {len(ollama_embedding[0])}")

shape: 4, 768


In [9]:
ollama_questions_embedding = ollama_client.embed(
    model="nomic-embed-text:latest",
    input=user_questions
)

questions_embedding = ollama_questions_embedding['embeddings']

print(f"shape: {len(questions_embedding)}, {len(questions_embedding[0])}")

shape: 3, 768


In [10]:
# finding the cosine similarity
similarity_scorer(ollama_embedding, questions_embedding)

User-Question: What is Python and what is it used for?
Similarity Score: 85.62% - Python is a high-level programming language
Similarity Score: 46.75% -  RAG stands for Retrieval-Augmented Generation
Similarity Score: 42.42% -  Vector databases store numerical representations called embeddings
Similarity Score: 53.69% - Machine learning is a field of artificial intelligence


User-Question: How does Retrieval-Augmented Generation (RAG) work?
Similarity Score: 44.73% - Python is a high-level programming language
Similarity Score: 80.39% -  RAG stands for Retrieval-Augmented Generation
Similarity Score: 53.23% -  Vector databases store numerical representations called embeddings
Similarity Score: 47.82% - Machine learning is a field of artificial intelligence


User-Question: What are embeddings and how do vector databases use them?
Similarity Score: 47.46% - Python is a high-level programming language
Similarity Score: 51.64% -  RAG stands for Retrieval-Augmented Generation
Similarity S

# Difference Between SentenceTransformer and Ollama for Embeddings

## 1. SentenceTransformer

- Uses pre-trained embedding models from the Sentence-Transformers library.
- Models are downloaded from Hugging Face and generally run locally.
- Directly converts text into numerical vector representations.
- Provides access to various embedding models such as `all-MiniLM-L6-v2`.
- Does not require an Ollama server.
- Offers flexibility for selecting, fine-tuning, and managing embedding models.

## 2. Ollama

- Uses locally hosted embedding models through the Ollama framework.
- Models must be downloaded using commands such as `ollama pull`.
- Generates embeddings through the Ollama API and Python SDK.
- Supports embedding models such as `nomic-embed-text`.
- Requires the Ollama server to be running.
- Allows centralized local model management and API-based access.

## 3. Key Difference

SentenceTransformer directly loads and executes embedding models through Python, whereas Ollama manages and serves models through a local server/API.

Both approaches generate vector embeddings that can be used for:

- Semantic Search
- Retrieval-Augmented Generation (RAG)
- Document Similarity
- Text Classification
- Vector Databases

## 4. Quick Comparison

| Feature | SentenceTransformer | Ollama |
|---|---|---|
| Execution | Directly through Python | Through Ollama server |
| Model source | Hugging Face | Ollama model library |
| Internet | Usually required for initial download | Usually required for initial download |
| Local execution | Yes | Yes |
| Server required | No | Yes |
| Model management | Python library | Ollama CLI/API |
| Embedding output | Numerical vectors | Numerical vectors |
| RAG support | Yes | Yes |

## 5. Recommendation

- **SentenceTransformer:** Best when you need direct Python integration, fine-grained control, and flexibility in selecting or fine-tuning embedding models.
- **Ollama:** Best when you want centralized local model management and an API-based workflow alongside locally hosted LLMs.

**Conclusion:** Both tools generate embeddings for semantic search and RAG applications. The primary difference is how embedding models are loaded, managed, and accessed.